In [3]:
import polars as pl

In [14]:
# create project root dir 
from pathlib import Path
project_root = Path.cwd().parent.resolve()


# reade cleaned budget parquet

data_dir = "data/processed"
df = pl.read_parquet(project_root / data_dir / "budget_cleaned.parquet")

# find unique values in type and fiscal_year columns using polars
print("Unique values in 'type' column:")
print(df.select(pl.col("type").unique()).to_series().to_list())
print("\nUnique values in 'fiscal_year' column:")
print(df.select(pl.col("fiscal_year").unique()).to_series().to_list())

Unique values in 'type' column:
['Budget - Governors Allowance', 'Budget - Actual', 'Budget - Working']

Unique values in 'fiscal_year' column:
[2020, 2021, 2023, 2024, 2025, 2026, 2027]


In [15]:
dim_df = df = pl.read_parquet(project_root / data_dir / "budget_dim.parquet")

# # export this as csv

dim_df.write_csv(project_root / data_dir /"budget_dim.csv")

In [ ]:
# load cost pool mappings YAML and join to subobject codes by code
import yaml

mapping_path = project_root / "configs" / "cost_pool_mappings.yaml"

with open(mapping_path, "r", encoding="utf-8") as f:
    mapping_cfg = yaml.safe_load(f) or {}

lookup = mapping_cfg.get("mappings", {})

subobject_df = pl.read_csv(project_root / "data" / "processed" / "subobject_codes.csv")

# normalize key type for robust joins
subobject_df = subobject_df.with_columns(
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars()
 )

mapping_rows = [
    {
        "comptroller_subobject_code": str(code).strip(),
        "cost_pool": attrs.get("cost_pool"),
        "opex_cost_post": attrs.get("cost_sub_pool"),
    }
    for code, attrs in lookup.items()
    if isinstance(attrs, dict)
]

mapping_df = pl.DataFrame(mapping_rows)

joined_df = subobject_df.join(mapping_df, on="comptroller_subobject_code", how="left")

print("Rows:", joined_df.height)
print("Mapped rows:", joined_df.filter(pl.col("cost_pool").is_not_null()).height)
print("Unmapped rows:", joined_df.filter(pl.col("cost_pool").is_null()).height)

joined_df.select(
    [   "object_code",
        "object_name",
        "comptroller_subobject_code",
        "comptroller_subobject_name",
        "cost_pool",
        "opex_cost_post",
    ]
).head(20)

Rows: 368
Mapped rows: 2
Unmapped rows: 366


object_code,object_name,comptroller_subobject_code,comptroller_subobject_name,cost_pool,opex_cost_post
i64,str,str,str,str,str
1,"""Salaries, Wages and Fringe Ben…","""152""","""Health Insurance""",null,"""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""112""","""Reclassifications""",null,"""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""105""","""Shift Differential""",null,"""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""151""","""Social Security Contributions""",null,"""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""110""","""Miscellaneous Adjustments""",null,"""Internal Labor"""
…,…,…,…,…,…
1,"""Salaries, Wages and Fringe Ben…","""111""","""Accrued Leave Payments""",null,"""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""175""","""Workers' Compensation""",null,"""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""102""","""Additional Assistance""",null,"""Internal Labor"""


In [23]:
# save csv in data/processed
joined_df.write_csv(project_root / data_dir / "subobject_codes.csv")

In [27]:
# join subprogram with tower classifications on organization_sub_code
subprogram_path = project_root / "data" / "processed" / "subprogram.csv"
tower_cls_path = project_root / "data" / "output" / "tower_classifications.csv"

subprogram_df = pl.read_csv(subprogram_path)
tower_df = pl.read_csv(tower_cls_path, encoding="utf8-lossy")

join_key = "organization_sub_code"

subprogram_df = subprogram_df.with_columns(
    pl.col(join_key).cast(pl.Utf8).str.strip_chars()
)
tower_df = tower_df.with_columns(
    pl.col(join_key).cast(pl.Utf8).str.strip_chars()
)

# keep only join key + last 2 columns from tower classifications
col_needed = tower_df.columns[-3:]
tower_keep_cols = [join_key, *col_needed]

joined_subprogram_tower_df = subprogram_df.join(
    tower_df.select(tower_keep_cols),
    on=join_key,
    how="left",
)

print("Rows:", joined_subprogram_tower_df.height)
print("Tower columns appended:", col_needed)

joined_subprogram_tower_df.head(20)


Rows: 178
Tower columns appended: ['tower', 'sub_tower', 'confidence']


organization_sub_code,agency_code,agency_name,unit_code,unit_name,program_code,program_name,subprogram_code,subprogram_name,organization_code,description,is_IT,IT_designination,tower,sub_tower,confidence
str,str,str,str,str,i64,str,str,str,str,str,bool,str,str,str,f64
"""C00_A00_12_T001""","""C00""","""Judiciary""","""A00""","""Judiciary""",12,"""Major Information Technology""","""T001""","""Case Management Modernization""","""C00_A00_12""","""The General Assembly adopted l…",true,"""MITDP""","""Application""","""Development""",0.9
"""C00_A00_12_T014""","""C00""","""Judiciary""","""A00""","""Judiciary""",12,"""Major Information Technology""","""T014""","""Courthouse eReadiness""","""C00_A00_12""","""The General Assembly adopted l…",true,"""MITDP""","""Application""","""Development""",0.8
"""C00_A00_12_T016""","""C00""","""Judiciary""","""A00""","""Judiciary""",12,"""Major Information Technology""","""T016""","""Cyber Security""","""C00_A00_12""","""The General Assembly adopted l…",true,"""MITDP""","""Security""","""Digital Security""",0.9
"""C00_A00_12_T018""","""C00""","""Judiciary""","""A00""","""Judiciary""",12,"""Major Information Technology""","""T018""","""Attorney Information Systems (…","""C00_A00_12""","""The General Assembly adopted l…",true,"""MITDP""","""Application""","""Support & Operations""",0.8
"""C00_A00_12_T019""","""C00""","""Judiciary""","""A00""","""Judiciary""",12,"""Major Information Technology""","""T019""","""Case Service V2""","""C00_A00_12""","""The General Assembly adopted l…",true,"""MITDP""","""Application""","""Development""",0.8
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""C00_A00_12_T074""","""C00""","""Judiciary""","""A00""","""Judiciary""",12,"""Major Information Technology""","""T074""","""Guardianship System""","""C00_A00_12""","""The General Assembly adopted l…",true,"""MITDP""","""Application""","""Development""",0.8
"""C00_A00_12_T075""","""C00""","""Judiciary""","""A00""","""Judiciary""",12,"""Major Information Technology""","""T075""","""VOIP – Enterprise Deployment P…","""C00_A00_12""","""The General Assembly adopted l…",true,"""MITDP""","""Network""","""Voice & Collaboration""",0.9
"""C00_A00_12_T076""","""C00""","""Judiciary""","""A00""","""Judiciary""",12,"""Major Information Technology""","""T076""","""Network Redesign""","""C00_A00_12""","""The General Assembly adopted l…",true,"""MITDP""","""Network""","""Network Management""",0.9


In [28]:
joined_subprogram_tower_df.write_csv(project_root / "data" / "processed" / "it_subprograms.csv")

In [ ]:
# join budget parquet with IT subprograms and subobject mappings

budget_path = project_root / "data" / "processed" / "budget_cleaned.parquet"
budget_df = pl.read_parquet(budget_path)

it_subprograms_path = project_root / "data" / "processed" / "it_subprograms.csv"
it_subprograms_df = pl.read_csv(it_subprograms_path)

subobject_codes_path = project_root / "data" / "processed" / "subobject_codes.csv"
subobject_codes_df = pl.read_csv(subobject_codes_path)

# Normalize join key types to avoid schema mismatches
budget_df = budget_df.with_columns(
    pl.col("organization_sub_code").cast(pl.Utf8).str.strip_chars(),
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars(),
)
it_subprograms_df = it_subprograms_df.with_columns(
    pl.col("organization_sub_code").cast(pl.Utf8).str.strip_chars()
)
subobject_codes_df = subobject_codes_df.with_columns(
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars()
)

# Keep only non-overlapping columns from right tables to prevent duplicate names
it_join_key = "organization_sub_code"
subobj_join_key = "comptroller_subobject_code"

it_right_cols = [
    c for c in it_subprograms_df.columns
    if c == it_join_key or c not in budget_df.columns
]

after_it_join = budget_df.join(
    it_subprograms_df.select(it_right_cols),
    on=it_join_key,
    how="left",
    suffix="_it",
)

subobj_right_cols = [
    c for c in subobject_codes_df.columns
    if c == subobj_join_key or c not in after_it_join.columns
]

joined_df = after_it_join.join(
    subobject_codes_df.select(subobj_right_cols),
    on=subobj_join_key,
    how="left",
    suffix="_subobj",
)

# Show joined output: all budget columns + appended enrichment columns
budget_cols = budget_df.columns
enrichment_cols = [c for c in joined_df.columns if c not in budget_cols]
final_cols = budget_cols + enrichment_cols
result_df = joined_df.select(final_cols)

print(f"Final joined shape: {result_df.shape}")
print(f"Budget columns: {len(budget_cols)}")
print(f"Enrichment columns appended: {len(enrichment_cols)}")

result_df.head(10)

DuplicateError: column with name 'agency_code_right' already exists

You may want to try:
- renaming the column prior to joining
- using the `suffix` parameter to specify a suffix different to the default one ('_right')